In [1]:
import torch
from torchvision.ops import box_iou
import numpy as np
from pathlib import Path
from torch.profiler import profile, ProfilerActivity, record_function
from typing import Callable, List, Tuple, Dict, Any, Sequence, Union, Optional, Iterable

import sys
sys.path.append("../..")
from v2.model_files.SSD_from_scratch import mySSD as mySSD
from v2.training_files.build_dataloaders import build_train_dl
from v2.training_files.CosSched import build_optimizer_and_scheduler
from v2.training_files.profile_training import profile_SSD_train, profile_SSD_train_2
from v2.training_files.profile_testing import profile_SSD_test, profile_SSD_test_2



device = "cuda" if torch.cuda.is_available() else "cpu"

# desktop, laptop, ubuntu
machine = 'ubuntu'

# Setup path to data folder
if machine == 'laptop':
    folder_path = Path(r"C:\self-driving-car\data")
elif machine == 'desktop':
    folder_path = Path(r"C:\Udacity_car_data\data")
elif machine == 'ubuntu':
    folder_path = Path(r"/mnt/c/Udacity_car_data/data")

train_path = folder_path / "train"
test_path = folder_path / "test"

In [2]:
ssdmodel = mySSD(class_to_idx_dict={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                 in_channels=3,
                 variances=(0.1, 0.2),
                 ).to(device)

train_dataloader, val_dataloader = build_train_dl(train_path=train_path,
                                                  batch_size=16,
                                                  num_workers=4,
                                                  prefetch_factor=4)

optimizer, scheduler = build_optimizer_and_scheduler(model=ssdmodel,
                                                     train_dataloader=train_dataloader,
                                                     max_epochs=150,
                                                     warmup_epochs=5,
                                                     base_lr=0.003,
                                                     min_lr=1e-6,
                                                     momentum=0.9,
                                                     weight_decay=0.005)

scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

In [5]:
profile_SSD_train(model=ssdmodel,
                  train_dataloader=train_dataloader,
                  test_dataloader=val_dataloader,
                  optimizer=optimizer,
                  scheduler=scheduler,
                  scaler=scaler,
                  sched_step_w_opt=False,
                  iou_thresh=0.5,
                  iou_variant="IoU",
                  neg_pos_ratio=3.0,
                  score_thresh=0.5,
                  nms_thresh=0.5,
                  max_detections_per_img=200,
                  epochs=1,
                  device=device)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             aten::convolution_backward         1.13%     140.460ms         2.94%     365.909ms     209.091us        3.865s        45.75%        4.001s       2.287ms           0 B           0 B      31.60 GB      29.15 G

In [4]:
# build_targets_2 on linux:
profile_SSD_train_2(model=ssdmodel,
                  train_dataloader=train_dataloader,
                  test_dataloader=val_dataloader,
                  optimizer=optimizer,
                  scheduler=scheduler,
                  scaler=scaler,
                  sched_step_w_opt=False,
                  iou_thresh=0.5,
                  iou_variant="IoU",
                  neg_pos_ratio=3.0,
                  score_thresh=0.5,
                  nms_thresh=0.5,
                  max_detections_per_img=200,
                  epochs=1,
                  device=device)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             aten::convolution_backward         1.35%     148.820ms         3.50%     385.267ms     220.153us        3.889s        46.02%        4.027s       2.301ms           0 B           0 B      31.60 GB      29.15 G

In [3]:
profile_SSD_test(model=ssdmodel,
                  test_dataloader=val_dataloader,
                  optimizer=optimizer,
                  scheduler=scheduler,
                  scaler=scaler,
                  sched_step_w_opt=False,
                  iou_thresh=0.5,
                  iou_variant="IoU",
                  neg_pos_ratio=3.0,
                  score_thresh=0.5,
                  nms_thresh=0.5,
                  max_detections_per_img=200,
                  epochs=1,
                  device=device)

/home/eblackstone/repos/ssd-venv/lib/python3.12/site-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         0.00%       0.000us         0.00%       0.000us       0.000us        2.744s        95.65%        2.744s      54.889ms           0 B           0 B           0 B           0 

In [4]:
# build_targets_2 on linux:
profile_SSD_test_2(model=ssdmodel,
                  test_dataloader=val_dataloader,
                  optimizer=optimizer,
                  scheduler=scheduler,
                  scaler=scaler,
                  sched_step_w_opt=False,
                  iou_thresh=0.5,
                  iou_variant="IoU",
                  neg_pos_ratio=3.0,
                  score_thresh=0.5,
                  nms_thresh=0.5,
                  max_detections_per_img=200,
                  epochs=1,
                  device=device)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         0.00%       0.000us         0.00%       0.000us       0.000us        2.728s        96.44%        2.728s      54.565ms           0 B           0 B           0 B           0 